In [28]:
# !pip install torchtext

In [2]:
# importing necessary libraries
import numpy as np
import pandas as pd
import sys,  torch, torch.nn as nn
from torch.utils.data import DataLoader,Dataset, TensorDataset
import torchtext
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import Vocab
from torch.nn.utils.rnn import pad_sequence
import tqdm
from collections import Counter
from time import perf_counter

In [3]:
import pandas as pd
csv_path="../dataset/poems-100.csv"

In [4]:
# tokenize:
text = "\n".join(pd.read_csv(csv_path)["text"].astype(str)).lower()
tokens = text.split()
vocab = sorted(set(tokens))
w2i = {w:i for i,w in enumerate(vocab)}
i2w = {i:w for w,i in w2i.items()}
encoded = [w2i[w] for w in tokens]

In [5]:
# sequence
seq_len = 6
X, Y = [], []
for i in range(len(encoded)-seq_len):
    X.append(encoded[i:i+seq_len])
    Y.append(encoded[i+seq_len])

In [6]:
# dataloader

X, Y = torch.tensor(X), torch.tensor(Y)
loader = DataLoader(TensorDataset(X,Y), batch_size=32, shuffle=True)

In [7]:
# Model:


class RNN(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.emb = nn.Embedding(vocab,64)
        self.rnn = nn.RNN(64,128,batch_first=True)
        self.fc = nn.Linear(128,vocab)

    def forward(self,x):
        x = self.emb(x)
        o,_ = self.rnn(x)
        return self.fc(o[:,-1])

In [8]:
model = RNN(len(vocab))
opt = torch.optim.Adam(model.parameters(),lr=0.003)
loss_fn = nn.CrossEntropyLoss()

In [10]:
# training
start=perf_counter()
for epoch in (range(20)):
    total=0
    for xb,yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb),yb)
        loss.backward()
        opt.step()
        total+=loss.item()
    print("epoch",epoch,"loss",total/len(loader))
end=perf_counter()
print(f"time taken:{end-start}")

epoch 0 loss 5.554291100273774
epoch 1 loss 4.341495063295648
epoch 2 loss 3.2535607197923153
epoch 3 loss 2.455420227593623
epoch 4 loss 1.9023122889258421
epoch 5 loss 1.50250553776345
epoch 6 loss 1.2237121201425136
epoch 7 loss 1.011616322926377
epoch 8 loss 0.8833167589959938
epoch 9 loss 0.7573426353129502
epoch 10 loss 0.674070215105702
epoch 11 loss 0.6213274354459706
epoch 12 loss 0.5560739016567596
epoch 13 loss 0.537568277945648
epoch 14 loss 0.5335684297522012
epoch 15 loss 0.5089619594396009
epoch 16 loss 0.5102571097201394
epoch 17 loss 0.48624922046474006
epoch 18 loss 0.4634190943684877
epoch 19 loss 0.46524261780574744
time taken:106.49634906999927


In [13]:
#  poem generation:
def generate(seed="love is",words=40):
    model.eval()
    toks = seed.lower().split()
    idxs = [w2i.get(w,0) for w in toks]

    for _ in range(words):
        x = torch.tensor([idxs[-seq_len:]])
        with torch.no_grad():
            p = torch.softmax(model(x),dim=-1)
        nxt = torch.multinomial(p,1).item()
        idxs.append(nxt)

    return " ".join(i2w[i] for i in idxs)

print("\nGenerated poem:\n")
print(generate())



Generated poem:

love is it? gray delicious according we night-time or with sorrow, seeming they and of your more the globe air. household no, my soul, something you tow-path, no apex walks race, does wider space, frozen when and thro' dusk—toss or through travels


Extra:
- save model weights, and model
- train on a larger dataset
- reduce the size of model
- show how data passes to model layers
- compare poem outputs with different training levels.